# Advanced RAG (Chunking, Reranking, Hybrid Search) Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: BM25 Implementation

In [ ]:
```python

import math

from collections import Counter

class BM25:

    def __init__(self, k1=1.2, b=0.75):

        self.k1 = k1

        self.b = b

        self.docs = []

        self.doc_lengths = []

        self.avg_dl = 0

        self.doc_freqs = {}

        self.n_docs = 0

    def index(self, documents):

        self.docs = documents

        self.n_docs = len(documents)

        self.doc_lengths = []

        self.doc_freqs = {}

        for doc in documents:

            words = doc.lower().split()

            self.doc_lengths.append(len(words))

            unique_words = set(words)

            for word in unique_words:

                self.doc_freqs[word] = self.doc_freqs.get(word, 0) + 1

        self.avg_dl = sum(self.doc_lengths) / self.n_docs if self.n_docs else 1

    def score(self, query, doc_idx):

        query_words = query.lower().split()

        doc_words = self.docs[doc_idx].lower().split()

        doc_len = self.doc_lengths[doc_idx]

        word_counts = Counter(doc_words)

        score = 0.0

        for term in query_words:

            if term not in word_counts:

                continue

            tf = word_counts[term]

            df = self.doc_freqs.get(term, 0)

            idf = math.log((self.n_docs - df + 0.5) / (df + 0.5) + 1)

            numerator = tf * (self.k1 + 1)

            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avg_dl)

            score += idf * numerator / denominator

        return score

    def search(self, query, top_k=10):

        scores = [(i, self.score(query, i)) for i in range(self.n_docs)]

        scores.sort(key=lambda x: x[1], reverse=True)

        return scores[:top_k]

In [ ]:
```

### Step 2: Reciprocal Rank Fusion

In [ ]:
```python

def reciprocal_rank_fusion(ranked_lists, k=60):

    scores = {}

    for ranked_list in ranked_lists:

        for rank, (doc_id, _) in enumerate(ranked_list):

            if doc_id not in scores:

                scores[doc_id] = 0.0

            scores[doc_id] += 1.0 / (k + rank + 1)

    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return fused

In [ ]:
```

### Step 3: Hybrid Search Pipeline

In [ ]:
```python

def hybrid_search(query, chunks, vector_embeddings, vocab, idf, bm25_index, top_k=5, fusion_k=60):

    query_emb = tfidf_embed(query, vocab, idf)

    vector_results = search(query_emb, vector_embeddings, top_k=top_k * 3)

    bm25_results = bm25_index.search(query, top_k=top_k * 3)

    fused = reciprocal_rank_fusion([vector_results, bm25_results], k=fusion_k)

    return fused[:top_k]

In [ ]:
```

### Step 4: Simple Reranker

In production, you would use a cross-encoder model. Here we build a reranker that scores query-document relevance using word overlap, term importance, and phrase matching.

In [ ]:
```python

def rerank(query, candidates, chunks):

    query_words = set(query.lower().split())

    stop_words = {"the", "a", "an", "is", "are", "was", "were", "what", "how",

                  "why", "when", "where", "do", "does", "for", "of", "in", "to",

                  "and", "or", "on", "at", "by", "it", "its", "this", "that",

                  "with", "from", "be", "has", "have", "had", "not", "but"}

    query_terms = query_words - stop_words

    scored = []

    for doc_id, initial_score in candidates:

        chunk = chunks[doc_id].lower()

        chunk_words = set(chunk.split())

        term_overlap = len(query_terms & chunk_words)

        query_bigrams = set()

        q_list = [w for w in query.lower().split() if w not in stop_words]

        for i in range(len(q_list) - 1):

            query_bigrams.add(q_list[i] + " " + q_list[i + 1])

        bigram_matches = sum(1 for bg in query_bigrams if bg in chunk)

        position_boost = 0

        for term in query_terms:

            pos = chunk.find(term)

            if pos != -1 and pos < len(chunk) // 3:

                position_boost += 0.5

        rerank_score = (

            term_overlap * 1.0

            + bigram_matches * 2.0

            + position_boost

            + initial_score * 5.0

        )

        scored.append((doc_id, rerank_score))

    scored.sort(key=lambda x: x[1], reverse=True)

    return scored

In [ ]:
```

### Step 5: HyDE (Hypothetical Document Embeddings)

In [ ]:
```python

def hyde_generate_hypothesis(query):

    templates = {

        "what": "The answer to '{query}' is as follows: Based on our documentation, {topic} involves specific policies and procedures that define how the process works.",

        "how": "To address '{query}': The process involves several steps. First, you need to initiate the request. Then, the system processes it according to the defined rules.",

        "default": "Regarding '{query}': Our records indicate specific details and policies related to this topic that provide a comprehensive answer."

    }

    query_lower = query.lower()

    if query_lower.startswith("what"):

        template = templates["what"]

    elif query_lower.startswith("how"):

        template = templates["how"]

    else:

        template = templates["default"]

    topic_words = [w for w in query.lower().split()

                   if w not in {"what", "is", "the", "how", "do", "does", "a", "an",

                                "for", "of", "to", "in", "on", "at", "by", "and", "or"}]

    topic = " ".join(topic_words) if topic_words else "this topic"

    return template.format(query=query, topic=topic)

def hyde_search(query, chunks, vector_embeddings, vocab, idf, top_k=5):

    hypothesis = hyde_generate_hypothesis(query)

    hypothesis_emb = tfidf_embed(hypothesis, vocab, idf)

    results = search(hypothesis_emb, vector_embeddings, top_k)

    return results, hypothesis

In [ ]:
```

### Step 6: Parent-Child Chunking

In [ ]:
```python

def create_parent_child_chunks(text, parent_size=200, child_size=50):

    words = text.split()

    parents = []

    children = []

    child_to_parent = {}

    parent_idx = 0

    start = 0

    while start < len(words):

        parent_end = min(start + parent_size, len(words))

        parent_text = " ".join(words[start:parent_end])

        parents.append(parent_text)

        child_start = start

        while child_start < parent_end:

            child_end = min(child_start + child_size, parent_end)

            child_text = " ".join(words[child_start:child_end])

            child_idx = len(children)

            children.append(child_text)

            child_to_parent[child_idx] = parent_idx

            child_start += child_size

        parent_idx += 1

        start += parent_size

    return parents, children, child_to_parent

In [ ]:
```

### Step 7: Faithfulness Evaluation

In [ ]:
```python

def evaluate_faithfulness(answer, retrieved_chunks):

    answer_sentences = [s.strip() for s in answer.split(".") if len(s.strip()) > 10]

    if not answer_sentences:

        return 1.0, []

    grounded = 0

    ungrounded = []

    context = " ".join(retrieved_chunks).lower()

    for sentence in answer_sentences:

        words = set(sentence.lower().split())

        stop_words = {"the", "a", "an", "is", "are", "was", "were", "and", "or",

                      "to", "of", "in", "for", "on", "at", "by", "it", "this", "that"}

        content_words = words - stop_words

        if not content_words:

            grounded += 1

            continue

        matched = sum(1 for w in content_words if w in context)

        ratio = matched / len(content_words) if content_words else 0

        if ratio >= 0.5:

            grounded += 1

        else:

            ungrounded.append(sentence)

    score = grounded / len(answer_sentences) if answer_sentences else 1.0

    return score, ungrounded

def evaluate_retrieval_recall(queries_with_relevant, retrieval_fn, k=5):

    total_recall = 0.0

    results = []

    for query, relevant_indices in queries_with_relevant:

        retrieved = retrieval_fn(query, k)

        retrieved_indices = set(idx for idx, _ in retrieved)

        relevant_set = set(relevant_indices)

        hits = len(retrieved_indices & relevant_set)

        recall = hits / len(relevant_set) if relevant_set else 1.0

        total_recall += recall

        results.append({

            "query": query,

            "recall": recall,

            "hits": hits,

            "total_relevant": len(relevant_set)

        })

    avg_recall = total_recall / len(queries_with_relevant) if queries_with_relevant else 0

    return avg_recall, results

In [ ]:
```

## Exercises

In [ ]:
1. Compare BM25 vs vector search vs hybrid search on the sample documents. For each of the 5 test queries, record which approach returns the most relevant chunk in position #1. Hybrid search should win on at least 3 out of 5.

2. Implement a metadata filter. Add a "category" field to each document (security, billing, api, product). Before running vector search, filter chunks to only the relevant category. Test with "What encryption is used?" and verify it only searches security-category chunks.

3. Build a full HyDE pipeline using the simple generate function from Lesson 06. Compare retrieval quality (top-3 relevance) between direct query search and HyDE search on all 5 test queries. HyDE should improve results for vague queries.

4. Implement the parent-child chunking strategy on the sample documents. Use child_size=30 and parent_size=100. Search with child chunks but return parent chunks in the prompt. Compare the generated answers to standard chunking with chunk_size=50.

5. Create an evaluation dataset: 10 questions with known answer chunks. Measure Recall@3, Recall@5, and Recall@10 for (a) vector search only, (b) BM25 only, (c) hybrid search, (d) hybrid + reranking. Plot the results and identify where reranking helps most.